In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/"

import os
os.chdir(r"C:\Z")  # Cambia el directorio de trabajo
print(os.getcwd())  # Verifica que cambió correctamente

In [ ]:
# TEST EXTRACCIÓN FITS

from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np

# Ruta al archivo FITS
file_path = r'TFGF NO DRIVE/Python/spectrums/spec-0266-51630-0003.fits'

# Leer el archivo FITS
with fits.open(file_path) as hdul:
    hdul.info()  # Información general del archivo

    # print(hdul[1].columns)  # Ver columnas del HDU 1
    test_flux = hdul[1].data["flux"]
    test_loglam = hdul[1].data["loglam"]

    # print(hdul[2].columns)  # Ver columnas del HDU 2
    test_redshift = hdul[2].data["Z"]
    print("Redshift:", test_redshift)
    test_class = hdul[2].data["CLASS"]
    print("Redshift:", test_class)

# Convertir loglam a longitud de onda (Ångstroms)
test_wavelength = 10 ** test_loglam

# Graficar el espectro
plt.figure(figsize=(12, 6))
plt.plot(test_wavelength, test_flux, label="Test Espectro")
plt.xlabel("Longitud de onda (Ångstrom)")
plt.ylabel("Flujo (10^-17 erg/s/cm²/Å)")
plt.title("Espectro vs. Flujo (Test)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
import os
import shutil
from astropy.io import fits

# Directorios de origen y destino
source_dir = r'TFGF NO DRIVE/Python/spectrums'
dest_dir = r'TFGF NO DRIVE/Python/spectrums2'

# Iterar sobre todos los archivos en el directorio de origen
for file_name in os.listdir(source_dir):
    if file_name.endswith('.fits'):
        file_path = os.path.join(source_dir, file_name)
        try:
            # Abrir el archivo FITS
            with fits.open(file_path) as hdul:
                # Extraer la clase desde el HDU 2
                file_class = hdul[2].data["CLASS"]
                # Si es un array, tomar el primer elemento
                if hasattr(file_class, '__iter__'):
                    file_class = file_class[0]
                # Convertir a cadena y normalizar
                file_class_str = str(file_class).strip().upper()
            
            # Verificar si la clase es "STAR"
            if file_class_str == 'STAR':
                dest_file_path = os.path.join(dest_dir, file_name)
                shutil.copy(file_path, dest_file_path)
                print(f"Archivo '{file_name}' copiado a '{dest_dir}'")
        except Exception as e:
            print(f"Error al procesar '{file_name}': {e}")

Archivo 'spec-0434-51885-0439.fits' copiado a 'TFGF NO DRIVE/Python/spectrums2'
Archivo 'spec-0434-51885-0463.fits' copiado a 'TFGF NO DRIVE/Python/spectrums2'
Archivo 'spec-0434-51885-0469.fits' copiado a 'TFGF NO DRIVE/Python/spectrums2'
Archivo 'spec-0434-51885-0481.fits' copiado a 'TFGF NO DRIVE/Python/spectrums2'
Archivo 'spec-0434-51885-0504.fits' copiado a 'TFGF NO DRIVE/Python/spectrums2'
Archivo 'spec-0434-51885-0505.fits' copiado a 'TFGF NO DRIVE/Python/spectrums2'
Archivo 'spec-0434-51885-0522.fits' copiado a 'TFGF NO DRIVE/Python/spectrums2'
Archivo 'spec-0434-51885-0523.fits' copiado a 'TFGF NO DRIVE/Python/spectrums2'
Archivo 'spec-0434-51885-0544.fits' copiado a 'TFGF NO DRIVE/Python/spectrums2'
Archivo 'spec-0434-51885-0550.fits' copiado a 'TFGF NO DRIVE/Python/spectrums2'
Archivo 'spec-0434-51885-0561.fits' copiado a 'TFGF NO DRIVE/Python/spectrums2'
Archivo 'spec-0434-51885-0563.fits' copiado a 'TFGF NO DRIVE/Python/spectrums2'
Archivo 'spec-0435-51882-0002.fits' copi

In [ ]:
# Función para interpolación adaptativa (15000 FITS = 5hrs POCO VIABLE EN DATASETS GRANDES PREPARAR LOS DATOS)

def expand_points(wavelength, flux, target_count=5000):
    # Convertimos a listas para facilitar las inserciones
    wl = list(wavelength)
    fl = list(flux)
    
    # Continuamos insertando hasta alcanzar el número deseado de puntos
    while len(wl) < target_count:
        # Calcular las diferencias absolutas en flux entre puntos consecutivos
        diffs = [abs(fl[i+1] - fl[i]) for i in range(len(fl) - 1)]
        # Encontrar el índice donde la diferencia es máxima
        max_idx = np.argmax(diffs)
        # Interpolar linealmente para obtener un nuevo punto
        new_wl = (wl[max_idx] + wl[max_idx+1]) / 2
        new_fl = (fl[max_idx] + fl[max_idx+1]) / 2
        # Insertar el nuevo punto en la posición correspondiente
        wl.insert(max_idx+1, new_wl)
        fl.insert(max_idx+1, new_fl)
        
    return np.array(wl), np.array(fl)

In [ ]:
def expand_points(wavelength, flux, target_count=5000):
    import numpy as np

    # Convertir a listas para facilitar las inserciones
    wl = list(wavelength)
    fl = list(flux)
    
    # Calcular las diferencias absolutas entre puntos consecutivos
    diffs = [abs(fl[i+1] - fl[i]) for i in range(len(fl)-1)]
    # Obtener los índices ordenados de mayor a menor diferencia
    sorted_indices = sorted(range(len(diffs)), key=lambda i: diffs[i], reverse=True)
    
    # Insertar nuevos puntos utilizando los índices ordenados
    while len(wl) < target_count:
        # Se recorre la lista de índices en orden descendente para evitar problemas con el reordenamiento
        for idx in sorted_indices:
            if len(wl) >= target_count:
                break
            # Calcular la interpolación lineal entre el punto idx y el siguiente
            new_wl = (wl[idx] + wl[idx+1]) / 2
            new_fl = (fl[idx] + fl[idx+1]) / 2
            # Insertar el nuevo punto en la posición correspondiente
            wl.insert(idx+1, new_wl)
            fl.insert(idx+1, new_fl)
    
    return np.array(wl), np.array(fl)

In [ ]:
# TEST CON EL ANTERIOR FITS
expanded_wavelength, expanded_flux = expand_points(test_wavelength, test_flux, target_count=5000)

# Graficar el espectro ampliado
plt.figure(figsize=(12, 6))
plt.plot(expanded_wavelength, expanded_flux, label="Espectro Expandido")
plt.xlabel("Longitud de onda (Ångstrom)")
plt.ylabel("Flujo (10^-17 erg/s/cm²/Å)")
plt.title("Espectro vs. Flujo (Expandido a 5000 puntos)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
import os
from astropy.io import fits
import numpy as np
import pickle

# APLICACIÓN A LOS ARCHIVOS DE LA CARPETA SPECTRUMS
folder_path = r'TFGF NO DRIVE/Python/spectrums'

# Diccionario de almacenamiento
spectra_data = {}

# Recorrer todos los archivos .fits en la carpeta
i=0
for filename in os.listdir(folder_path):
    if filename.endswith('.fits'):
        file_path = os.path.join(folder_path, filename)
        try:
            with fits.open(file_path) as hdul:
                # Verifica que el archivo tenga las extensiones esperadas
                if len(hdul) > 2:
                    flux_data = hdul[1].data["flux"]         # Datos de flujo
                    loglam_data = hdul[1].data["loglam"]     # Datos de log(lambda)
                    wavelength_data = 10 ** loglam_data      # Convertir log(lambda) a longitud de onda
                    redshift = hdul[2].data["Z"][0]          # Extraer redshift
                    
                    # Aplicar la interpolación para obtener 5000 puntos
                    interp_wavelength, interp_flux = expand_points(wavelength_data, flux_data, target_count=5000)
                    
                    # Guardar los datos
                    spectra_data[filename] = {
                        "wavelength": interp_wavelength,
                        "flux": interp_flux,
                        "redshift": redshift
                    }

                    i=i+1
                    print(f"Procesado {filename} ({i})")
                    
        except Exception as e:
            print(f"Error procesando {filename}: {e}")

# Imprimir un resumen de los datos extraídos
print(f"Se procesaron {len(spectra_data)} archivos FITS.")

In [ ]:
# GUARDAR LOS DATOS EN UN ARCHIVO PICKLE PARA USAR EN OTRO NOTEBOOK

output_dir = r'TFGF NO DRIVE/Python/storage'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_file = os.path.join(output_dir, 'spectra_data_full2.pkl')

# Guardar spectra_data en un archivo pickle para usar en otro notebook
try:
    with open(output_file, 'wb') as f:
        pickle.dump(spectra_data, f)
    print(f"Spectra data guardada exitosamente en {output_file}")
except Exception as e:
    print(f"Error al guardar spectra_data: {e}")

In [ ]:
# COMRPOBAR QUE LOS DATOS TIENEN LAS DIMENSIONES CORRECTAS

# Variable para almacenar el máximo
min_flux_length = 0
max_flux_length = 0

# Recorrer los datos de cada archivo en el diccionario spectra_data
for filename, data in spectra_data.items():
    flux_length = len(data["flux"])  # Longitud del array de flujo

    if flux_length > max_flux_length:
        max_flux_length = flux_length

    if min_flux_length > flux_length:
        min_flux_length = flux_length

# Imprimir la longitud máxima encontrada
print(f"\nLa longitud máxima de flujo encontrada es: {max_flux_length}")